# Headline Dataset Preprocessing

In [3]:
# ==========================================
# HEADLINES DATASET - EXPLORATORY DATA ANALYSIS
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import os
import re
import string

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [6]:
# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv('articles_headlines.csv')

In [7]:
# ==========================================
# DATASET OVERVIEW
# ==========================================

print("="*60)
print("DATASET INFORMATION")
print("="*60)

display(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nData Types")
display(df.dtypes)

print("\nDataset Info")
df.info()

print("\nDescriptive Statistics")
display(df.describe(include="all"))

DATASET INFORMATION


,headline,clickbait
0,Should I Get Bings,1
1,Which TV Female Friend Group Do You Belong In,1
2,"The New ""Star Wars: The Force Awakens"" Trailer...",1
3,"This Vine Of New York On ""Celebrity Big Brothe...",1
4,A Couple Did A Stunning Photo Shoot With Their...,1



Dataset Shape
(31996, 2)

Columns
['headline', 'clickbait']

Data Types


headline     object
clickbait     int64
dtype: object


Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31996 entries, 0 to 31995
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   headline   31996 non-null  object
 1   clickbait  31996 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 500.1+ KB

Descriptive Statistics


,headline,clickbait
count,31996,31996.000000
unique,31996,NaN
top,Should I Get Bings,NaN
freq,1,NaN
mean,NaN,0.499906
std,NaN,0.500008
min,NaN,0.000000
25%,NaN,0.000000
50%,NaN,0.000000
75%,NaN,1.000000


In [8]:
# ==========================================
# MISSING VALUES
# ==========================================

missing = df.isnull().sum()

missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Percentage":
    (missing/len(df))*100
})

display(missing_df)


,Missing Values,Percentage
headline,0,0.0
clickbait,0,0.0


In [9]:
# ==========================================
# DUPLICATES
# ==========================================

duplicates = df.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

headline_duplicates = df["headline"].duplicated().sum()

print(f"Duplicate Headlines : {headline_duplicates}")

Duplicate Rows : 0
Duplicate Headlines : 0


## Data Preprocessing

In [10]:
# ===========================
# text cleaning function
# ==========================

def clean_text(text):
  # lowercase
  text = str(text).lower()
  # remove URLs
  text = re.sub(r'http\S+\s+', ' ', text)
  # remove html tags
  text = re.sub(r'<.*?>', ' ', text)
  # remove punctuation
  text = text.translate(str.maketrans('', '', string.punctuation))
  # remove numbers
  text = re.sub(r'\d+', ' ', text)
  # remove extra spaces
  text = re.sub(r'\s+', ' ', text)
  return text


In [11]:
df['headline'] = df['headline'].apply(clean_text)

In [12]:
df['headline'].head()

0                                   should i get bings
1        which tv female friend group do you belong in
2    the new star wars the force awakens trailer is...
3    this vine of new york on celebrity big brother...
4    a couple did a stunning photo shoot with their...
Name: headline, dtype: object

## Dataset Splitting (70/15/15)

In [13]:
# =================================
# Separate features (X) and labels
# =================================

X = df["headline"]

y = df["clickbait"]

In [14]:
# =========================================
# First Split into train and temporary data
# =========================================


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [15]:
# ==========================================
# Second split into validation and test data
# ==========================================

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)


In [16]:
# ================================
# verify shapes of the data splits
# ================================

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (22397,)
Validation: (4799,)
Testing: (4800,)


## Tokenizer Fitting on Train Data

In [17]:
# ======================
# Applying tokenization
# ======================

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 20278


## Convert Text to Token Sequences

In [18]:
# =================================
# Converting headlines to sequences
# =================================

train_sequences = tokenizer.texts_to_sequences(X_train)
val_sequences=tokenizer.texts_to_sequences(X_val)
test_sequences=tokenizer.texts_to_sequences(X_test)

## Apply Padding to Token Sequences

In [19]:
lengths = [len(seq) for seq in train_sequences]
max_sequence_length =  int(np.percentile(lengths,95))
avg_sequence_length = np.mean(lengths)
print("Maximum sequence length:", max_sequence_length)
print("Average sequence length:", avg_sequence_length)

Maximum sequence length: 13
Average sequence length: 8.756887083091485


In [20]:
# ==================
# Padding Train Data
# ==================

X_train = pad_sequences(
    train_sequences,
    maxlen = max_sequence_length,
    padding = 'post',
    truncating = 'post'
)

In [21]:
# =======================
# Padding Validation Data
# ======================

X_val=pad_sequences(

    val_sequences,

    maxlen = max_sequence_length,

    padding = "post",

    truncating = "post"
)

In [22]:
# ==================
# Padding Test Data
# ==================

X_test=pad_sequences(

    test_sequences,

    maxlen = max_sequence_length,

    padding = "post",

    truncating = "post"
)

## Convert Labels to NumPy Arrays

In [23]:
y_train = np.array(y_train)

y_val = np.array(y_val)

y_test = np.array(y_test)

In [24]:
# ========================================================
# Verify Shapes of the features and labels of all datasets
# ========================================================

print(X_train.shape)

print(X_val.shape)

print(X_test.shape)

print(y_train.shape)

print(y_val.shape)

print(y_test.shape)

(22397, 13)
(4799, 13)
(4800, 13)
(22397,)
(4799,)
(4800,)


In [26]:
# =============================
# Saving the Headline Tokenizer
# =============================

import pickle
import os

# Define the local path for saving in the notebook environment
local_path = 'E:/FYP PROJECT'  # Current directory where the notebook is running

# Ensure the directory exists (optional, but good practice)
os.makedirs(local_path, exist_ok=True)

# Define the full path for the tokenizer file locally
tokenizer_save_path = os.path.join(local_path, "headline_tokenizer.pkl")

# Save the tokenizer with error handling
try:
    with open(tokenizer_save_path, "wb") as f:
        pickle.dump(tokenizer, f)
    print(f"Tokenizer successfully saved locally at '{tokenizer_save_path}'")
    
    # Verify the file was created and show its size
    file_size = os.path.getsize(tokenizer_save_path)
    print(f"File size: {file_size:,} bytes")
    
    # Show current directory contents to confirm file exists
    print("\nFiles in current directory:")
    for file in os.listdir(local_path):
        if file.endswith('.pkl'):
            print(f"- {file}")
    
except Exception as e:
    print(f"Error saving tokenizer: {e}")

# To load the tokenizer later in the same session, you can use:
# with open(tokenizer_save_path, "rb") as f:
#     loaded_tokenizer = pickle.load(f)

Tokenizer successfully saved locally at 'E:/FYP PROJECT\headline_tokenizer.pkl'
File size: 812,456 bytes

Files in current directory:
- headline_tokenizer.pkl


In [27]:
# =============================
# Saving all the Datasets
# =============================

import numpy as np
import os
from typing import Any, Optional

def save_datasets_to_path(
    base_path: str,
    X_train: np.ndarray,
    X_val: np.ndarray, 
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_val: np.ndarray,
    y_test: np.ndarray,
    create_dir: bool = True
) -> None:
    
    if create_dir:
        os.makedirs(base_path, exist_ok=True)
    
    # Dataset filenames mapping
    datasets = {
        'X_train.npy': X_train,
        'X_val.npy': X_val,
        'X_test.npy': X_test,
        'y_train.npy': y_train,
        'y_val.npy': y_val,
        'y_test.npy': y_test
    }
    
    # Save all datasets
    for filename, data in datasets.items():
        file_path = os.path.join(base_path, filename)
        np.save(file_path, data)
    
    print(f"All training, validation, and test datasets saved successfully to {base_path}")

def load_datasets_from_path(base_path: str) -> tuple[np.ndarray, ...]:
    
    filenames = ['X_train.npy', 'X_val.npy', 'X_test.npy', 
                'y_train.npy', 'y_val.npy', 'y_test.npy']
    
    datasets = []
    for filename in filenames:
        file_path = os.path.join(base_path, filename)
        datasets.append(np.load(file_path))
    
    return tuple(datasets)

# Define the local path for saving in the notebook environment
local_path = 'E:/FYP PROJECT'  # Current directory where the notebook is running

# Ensure the directory exists (optional, but good practice)
os.makedirs(local_path, exist_ok=True)

# Define the full path for the tokenizer file locally
datasets_path = os.path.join(local_path)

save_datasets_to_path(local_path, X_train, X_val, X_test, y_train, y_val, y_test)

All training, validation, and test datasets saved successfully to E:/FYP PROJECT


# YouTube Dataset Preprocessing

In [1]:
# ====================================================
# Import all required libraries
# ====================================================

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import re
import string
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import load_img, img_to_array


In [2]:
#=========================
# Load the YouTube Dataset
# ========================

df = pd.read_csv("final_research_dataset.csv")

In [3]:
# ==========================================
# DATASET OVERVIEW
# ==========================================

print("="*60)
print("DATASET INFORMATION")
print("="*60)

display(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nData Types")
display(df.dtypes)

print("\nDataset Info")
df.info()

print("\nDescriptive Statistics")
display(df.describe())

DATASET INFORMATION


,video_id,title,description,thumbnail_url,thumbnail_path,view_count,like_count,comment_count,duration_seconds,channel_title,channel_subscribers,likes_per_view,comments_per_view,top_5_comments,video_category,tags,clickbait
0,hxwpkM5w3Cc,I Got Hunted By The FBI,New Merch - https://mrbeast.store\n\nCheck out...,https://i.ytimg.com/vi/hxwpkM5w3Cc/hqdefault.jpg,thumbnails_SUSPECTED\hxwpkM5w3Cc.jpg,264619555,4860925,94400,951,MrBeast,469000000,0.018369,0.000357,Subscribe for a free car! ||| He didn't smile ...,Entertainment,NaN,0
1,ZZtYAOpHVYk,10 Real Life Giants,Top 10 Real Life GIANTS\nSubscribe to Top 10s ...,https://i.ytimg.com/vi/ZZtYAOpHVYk/hqdefault.jpg,thumbnails_SUSPECTED\ZZtYAOpHVYk.jpg,9248337,63045,3666,626,Top 10s,5160000,0.006817,0.000396,LOOK AT THE LADY IN THE BACKGROUNDS TONGUE AT ...,Education,"giants, real, fake, real life, massive, bigges...",0
2,M6vNwTQ1n5U,30 BEST PRANKS AND FUNNY TRICKS FOR YOUR FRIENDS,Hilarious prank ideas and trick to pull on you...,https://i.ytimg.com/vi/M6vNwTQ1n5U/hqdefault.jpg,thumbnails_SUSPECTED\M6vNwTQ1n5U.jpg,5823198,42736,1070,629,5-Minute Crafts,80800000,0.007339,0.000184,I just did the most satisfying thing... I chan...,Howto & Style,"100 layers, 100 layers of clothes, 100 layers ...",1
3,zxYjTTXc-J8,"Last To Leave Circle Wins $500,000",THIS WAS THE CRAZIEST THING IVE EVER DONE!\n\n...,https://i.ytimg.com/vi/zxYjTTXc-J8/hqdefault.jpg,thumbnails_SUSPECTED\zxYjTTXc-J8.jpg,549979630,7857388,130366,1064,MrBeast,469000000,0.014287,0.000237,Subscribe and you could be flown down for one ...,Entertainment,NaN,0
4,1iNoQSM8pqU,this video makes you forget your name..,this video makes you forget your name.. In thi...,https://i.ytimg.com/vi/1iNoQSM8pqU/hqdefault.jpg,thumbnails_SUSPECTED\1iNoQSM8pqU.jpg,970169,73062,5048,499,Adventure,2800000,0.075309,0.005203,"(Hallway illusion)\nMe: oh, it's right\nMy min...",Gaming,"this video will make you forget your name.., f...",0



Dataset Shape
(30256, 17)

Columns
['video_id', 'title', 'description', 'thumbnail_url', 'thumbnail_path', 'view_count', 'like_count', 'comment_count', 'duration_seconds', 'channel_title', 'channel_subscribers', 'likes_per_view', 'comments_per_view', 'top_5_comments', 'video_category', 'tags', 'clickbait']

Data Types


video_id                object
title                   object
description             object
thumbnail_url           object
thumbnail_path          object
view_count               int64
like_count               int64
comment_count            int64
duration_seconds         int64
channel_title           object
channel_subscribers      int64
likes_per_view         float64
comments_per_view      float64
top_5_comments          object
video_category          object
tags                    object
clickbait                int64
dtype: object


Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30256 entries, 0 to 30255
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   video_id             30256 non-null  object 
 1   title                30256 non-null  object 
 2   description          29283 non-null  object 
 3   thumbnail_url        30256 non-null  object 
 4   thumbnail_path       30237 non-null  object 
 5   view_count           30256 non-null  int64  
 6   like_count           30256 non-null  int64  
 7   comment_count        30256 non-null  int64  
 8   duration_seconds     30256 non-null  int64  
 9   channel_title        30256 non-null  object 
 10  channel_subscribers  30256 non-null  int64  
 11  likes_per_view       30256 non-null  float64
 12  comments_per_view    30256 non-null  float64
 13  top_5_comments       18603 non-null  object 
 14  video_category       30256 non-null  object 
 15  tags                 2

,view_count,like_count,comment_count,duration_seconds,channel_subscribers,likes_per_view,comments_per_view,clickbait
count,3.025600e+04,3.025600e+04,30256.000000,30256.000000,3.025600e+04,30256.000000,30256.000000,30256.000000
mean,1.351214e+06,2.257652e+04,979.031432,1036.981987,2.473651e+07,0.021204,0.002306,0.509750
std,8.439158e+06,1.708166e+05,3883.557253,2155.711250,2.804577e+07,0.167879,0.046059,0.499913
min,0.000000e+00,0.000000e+00,0.000000,0.000000,2.000000e+01,0.000000,0.000000,0.000000
25%,3.225825e+04,4.910000e+02,25.000000,305.000000,2.920000e+06,0.009648,0.000379,0.000000
50%,1.629055e+05,2.635000e+03,202.000000,610.000000,1.630000e+07,0.016800,0.001264,1.000000
75%,6.168918e+05,9.249250e+03,750.250000,898.000000,2.590000e+07,0.025354,0.002752,1.000000
max,5.499796e+08,1.139868e+07,224105.000000,61764.000000,4.690000e+08,29.000000,8.000000,1.000000


In [82]:
# ===============
# Missing Values
# ==============

missing=df.isnull().sum()

missing_df=pd.DataFrame({

"Missing Values":missing,

"Percentage":round(missing/len(df)*100,2)

})

display(missing_df)

,Missing Values,Percentage
video_id,0,0.00
title,0,0.00
description,973,3.22
thumbnail_url,0,0.00
thumbnail_path,19,0.06
view_count,0,0.00
like_count,0,0.00
comment_count,0,0.00
duration_seconds,0,0.00
channel_title,0,0.00


In [83]:
# ================
# Duplicate Values
# ================

print("Duplicate Rows :",df.duplicated().sum())

print("Duplicate Titles :",df["title"].duplicated().sum())

Duplicate Rows : 0
Duplicate Titles : 0


In [84]:
# =======================
# Impute Missing Values
# ======================

df["description"] = df["description"].fillna("")

df["tags"] = df["tags"].fillna("")

df["top_5_comments"] = df["top_5_comments"].fillna("")

In [85]:
# ========================================
# Remove Rows with Missing Thumbnail Paths
# ========================================

df = df.dropna(subset=["thumbnail_path"]).reset_index(drop=True)

print(df.shape)

(30237, 17)


In [86]:
# ================
# Verifying Values
# ================

df.isna().sum()

video_id               0
title                  0
description            0
thumbnail_url          0
thumbnail_path         0
view_count             0
like_count             0
comment_count          0
duration_seconds       0
channel_title          0
channel_subscribers    0
likes_per_view         0
comments_per_view      0
top_5_comments         0
video_category         0
tags                   0
clickbait              0
dtype: int64

In [87]:
# ===========================
# Remove Unncessary Columns
# ===========================

df = df.drop(columns=[
    "video_id",
    "thumbnail_url"
], errors='ignore')

In [88]:
# ===============================================
# Combine all text features into a single feature
# ===============================================

df["text"] = (
    "TITLE" + " " + df["title"].astype(str) + " " +
    "DESCRIPTION" + " " + df["description"].astype(str) + " " +
    "TAGS" + " " + df["tags"].astype(str) + " " +
    "COMMENTS" + " " + df["top_5_comments"].astype(str) + " "+
    "CATEGORY" + " " + df["video_category"].astype(str) + " " +
    "CHANNEL" + " " + df["channel_title"].astype(str)
)

In [89]:
df['text']

0        TITLE I Got Hunted By The FBI DESCRIPTION New ...
1        TITLE 10 Real Life Giants DESCRIPTION Top 10 R...
2        TITLE 30 BEST PRANKS AND FUNNY TRICKS FOR YOUR...
3        TITLE Last To Leave Circle Wins $500,000 DESCR...
4        TITLE this video makes you forget your name.. ...
                               ...                        
30232    TITLE Barometer Example Problem #2 DESCRIPTION...
30233    TITLE Hydraulic Lift Example Problem DESCRIPTI...
30234    TITLE Pascals Law DESCRIPTION This tutorial in...
30235    TITLE Integration By Parts the FAST Way - Exam...
30236    TITLE Integration By Parts the FAST Way - Exam...
Name: text, Length: 30237, dtype: object

In [90]:
print(df.columns)

Index(['title', 'description', 'thumbnail_path', 'view_count', 'like_count',
       'comment_count', 'duration_seconds', 'channel_title',
       'channel_subscribers', 'likes_per_view', 'comments_per_view',
       'top_5_comments', 'video_category', 'tags', 'clickbait', 'text'],
      dtype='object')


## Data Preprocessing

### Text Branch

In [91]:
# ======================
# Text Cleaning Function
# ======================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"<.*?>", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

### Metadata Branch

In [92]:
df['text'] = df['text'].apply(clean_text)

In [93]:
metadata = df[[
    "view_count",
    "like_count",
    "comment_count",
    "duration_seconds",
    "channel_subscribers",
    "likes_per_view",
    "comments_per_view"
]]

In [94]:
# ==============
# Missing Values
# ==============

print("Missing values:\n", metadata.isna().sum())

Missing values:
 view_count             0
like_count             0
comment_count          0
duration_seconds       0
channel_subscribers    0
likes_per_view         0
comments_per_view      0
dtype: int64


In [95]:
# ======================
# Separate Binary Labels
# ======================

labels = df["clickbait"]

In [96]:
# ======================================================================
# Replace "\\" with "/" ensure consistent file path formatting across OS
# ======================================================================

import os

BASE_DIR = r"E:/FYP PROJECT"

df["thumbnail_path"] = df["thumbnail_path"].str.replace("\\", "/", regex=False)

df["thumbnail_path"] = df["thumbnail_path"].apply(
    lambda x: os.path.join(BASE_DIR, x)
)

In [97]:
image_path = df["thumbnail_path"]

## Data Splitting (70/15/15)

In [98]:
# =================================================
# First Split into Train and Temporary Data (70/30)
# =================================================

X_text_train, X_text_temp, X_meta_train, X_meta_temp, X_img_train, X_img_temp, y_train, y_temp = train_test_split(

    df["text"],

    metadata,

    image_path,

    labels,

    test_size=0.30,

    random_state=42,

    stratify=labels
)

In [99]:
# ==================================================
# Second Split into Validation and Test Data (50/50)
# ==================================================

X_text_val, X_text_test, X_meta_val, X_meta_test, X_img_val, X_img_test, y_val, y_test = train_test_split(

    X_text_temp,

    X_meta_temp,

    X_img_temp,

    y_temp,

    test_size=0.50,

    random_state=42,

    stratify=y_temp
)

In [103]:
# Reset text indices
X_text_train = X_text_train.reset_index(drop=True)
X_text_val   = X_text_val.reset_index(drop=True)
X_text_test  = X_text_test.reset_index(drop=True)

# Reset metadata indices
X_meta_train = X_meta_train.reset_index(drop=True)
X_meta_val   = X_meta_val.reset_index(drop=True)
X_meta_test  = X_meta_test.reset_index(drop=True)

# Reset image path indices
X_img_train = X_img_train.reset_index(drop=True)
X_img_val   = X_img_val.reset_index(drop=True)
X_img_test  = X_img_test.reset_index(drop=True)

# Reset label indices
y_train = y_train.reset_index(drop=True)
y_val   = y_val.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

In [105]:
# =========================
# Verify Length of Features
# =========================

print(len(X_text_train))
print(len(X_text_val))
print(len(X_text_test))

21165
4536
4536


## BRANCH 1: Text Processing (BiLSTM)

In [106]:
# ==============================
# Tokenizer Fit on Training Data
# ==============================

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_text_train)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 130751


### Convert Text to Token Sequences

In [107]:
train_sequences = tokenizer.texts_to_sequences(X_text_train)

val_sequences = tokenizer.texts_to_sequences(X_text_val)

test_sequences = tokenizer.texts_to_sequences(X_text_test)

### Convert Text to Token Sequences

In [108]:
train_lengths = [len(seq) for seq in train_sequences]

MAX_LENGTH = int(np.percentile(train_lengths,95))

print("Maximum Length :",MAX_LENGTH)

Maximum Length : 599


### Apply Padding to Sequences

In [109]:
X_text_train = pad_sequences(

    train_sequences,

    maxlen = MAX_LENGTH,

    padding = "post",

    truncating = "post"
)

In [110]:
X_text_val = pad_sequences(

    val_sequences,

    maxlen = MAX_LENGTH,

    padding = "post",

    truncating = "post"
)

In [111]:
X_text_test = pad_sequences(

    test_sequences,

    maxlen = MAX_LENGTH,

    padding = "post",

    truncating = "post"
)

In [112]:
# ======================
# Save youTube Tokenizer
# ======================


import pickle
import os


# Define the base path for saving in Google Drive (using previous path)
drive_path = r'E:/FYP PROJECT'

# Ensure the directory exists (optional, but good practice)
os.makedirs(drive_path, exist_ok=True)

# Define the full path for the tokenizer file in Google Drive
tokenizer_save_path = os.path.join(drive_path, "youtube_text_tokenizer.pkl")

with open(tokenizer_save_path, "wb") as f:
    pickle.dump(tokenizer, f)

print(f"Tokenizer successfully saved at '{tokenizer_save_path}'")

Tokenizer successfully saved at 'E:/FYP PROJECT\youtube_text_tokenizer.pkl'


## BRANCH 2: Metadata Processing

In [113]:
# =========================
# Standardize the metadata
# ========================

scaler = StandardScaler()
X_meta_train = scaler.fit_transform(X_meta_train)
X_meta_val = scaler.transform(X_meta_val)
X_meta_test = scaler.transform(X_meta_test)

In [114]:
# =======================================
# Saving the standard scaler for metadata
# =======================================

# Define the base path for saving in Google Drive (using previous path)
drive_path = r'E:/FYP PROJECT'

# Ensure the directory exists (optional, but good practice)
os.makedirs(drive_path, exist_ok=True)

# Define the full path for the tokenizer file in Google Drive
tokenizer_save_path = os.path.join(drive_path, "youtube_metadata_scaler.pkl")

with open(tokenizer_save_path, "wb") as f:
    pickle.dump(scaler, f)

print(f"Scaler successfully saved at '{tokenizer_save_path}'")

Scaler successfully saved at 'E:/FYP PROJECT\youtube_metadata_scaler.pkl'


## BRANCH 3: Thumbnail Processing

In [115]:
import os
import numpy as np
from PIL import Image

In [116]:
# =====================
# Define the Image Size
# =====================

IMG_SIZE = (224, 224)

In [117]:
# ============================
# Image Preprocessing Function
# ============================

def preprocess_image(image_path):

    try:
        image = Image.open(image_path)

        # Convert grayscale/RGBA to RGB
        image = image.convert("RGB")

        # Resize image
        image = image.resize(IMG_SIZE)

        # Convert to NumPy array
        image = np.array(image, dtype=np.float32)

        # Normalize pixel values
        image = image / 255.0

        return image

    except Exception as e:
        print(f"Error loading {image_path}: {e}")

        # Return a blank image if loading fails
        return np.zeros((224, 224, 3), dtype=np.float32)

In [60]:
# ==========================
# Remove invalid image paths
# ==========================

df = df[df["thumbnail_path"].apply(os.path.exists)].reset_index(drop=True)

print("Remaining samples:", len(df))

Remaining samples: 30237


In [118]:
# ======================================== 
# Process and Save Train Images in Batches
# ========================================

import os
import numpy as np

# Folder to save batches
SAVE_DIR = r"E:/FYP PROJECT/processed_train_batches"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64

# Total number of batches
total_batches = int(np.ceil(len(X_img_train) / BATCH_SIZE))

print(f"Total batches to create: {total_batches}")

# Process ALL batches from 0 to last batch
for batch_idx in range(total_batches):

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(X_img_train))

    batch_paths = X_img_train.iloc[start:end]

    print(f"Processing Batch {batch_idx + 1}/{total_batches}...")

    batch_images = np.array(
        [preprocess_image(path) for path in batch_paths],
        dtype=np.float32
    )

    save_path = os.path.join(
        SAVE_DIR,
        f"train_batch_{batch_idx + 1}.npy"
    )

    np.save(save_path, batch_images)

    print(f"Saved {save_path} ({batch_images.shape[0]} images)")

    # Free RAM before processing next batch
    del batch_images

print("\nAll training batches have been saved successfully!")

Total batches to create: 331
Processing Batch 1/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_1.npy (64 images)
Processing Batch 2/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_2.npy (64 images)
Processing Batch 3/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_3.npy (64 images)
Processing Batch 4/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_4.npy (64 images)
Processing Batch 5/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_5.npy (64 images)
Processing Batch 6/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_6.npy (64 images)
Processing Batch 7/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_7.npy (64 images)
Processing Batch 8/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_8.npy (64 images)
Processing Batch 9/331...
Saved E:/FYP PROJECT/processed_train_batches\train_batch_9.npy (64 images)
Processing Batch 10/331...
Saved E:/FYP PROJECT/processed_trai

In [119]:
# ============================================= 
# Process and Save Validation Images in Batches
# =============================================

import os
import numpy as np

# Folder to save batches
SAVE_DIR = r"E:/FYP PROJECT/processed_val_batches"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64

# Total number of batches
total_batches = int(np.ceil(len(X_img_val) / BATCH_SIZE))

print(f"Total batches to create: {total_batches}")

# Process ALL batches from 0 to last batch
for batch_idx in range(total_batches):

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(X_img_val))

    batch_paths = X_img_val.iloc[start:end]

    print(f"Processing Batch {batch_idx + 1}/{total_batches}...")

    batch_images = np.array(
        [preprocess_image(path) for path in batch_paths],
        dtype=np.float32
    )

    save_path = os.path.join(
        SAVE_DIR,
        f"val_batch_{batch_idx + 1}.npy"
    )

    np.save(save_path, batch_images)

    print(f"Saved {save_path} ({batch_images.shape[0]} images)")

    # Free RAM before processing next batch
    del batch_images

print("\nAll validation batches have been saved successfully!")

Total batches to create: 71
Processing Batch 1/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_1.npy (64 images)
Processing Batch 2/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_2.npy (64 images)
Processing Batch 3/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_3.npy (64 images)
Processing Batch 4/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_4.npy (64 images)
Processing Batch 5/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_5.npy (64 images)
Processing Batch 6/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_6.npy (64 images)
Processing Batch 7/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_7.npy (64 images)
Processing Batch 8/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_8.npy (64 images)
Processing Batch 9/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_9.npy (64 images)
Processing Batch 10/71...
Saved E:/FYP PROJECT/processed_val_batches\val_batch_10.npy (64 images)
Processing

In [121]:
# ======================================== 
# Process and Save Test Images in Batches
# ========================================

import os
import numpy as np

# Folder to save batches
SAVE_DIR = r"E:/FYP PROJECT/processed_test_batches"
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64

# Total number of batches
total_batches = int(np.ceil(len(X_img_test) / BATCH_SIZE))

print(f"Total batches to create: {total_batches}")

# Process ALL batches from 0 to last batch
for batch_idx in range(total_batches):

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(X_img_test))

    batch_paths = X_img_test.iloc[start:end]

    print(f"Processing Batch {batch_idx + 1}/{total_batches}...")

    batch_images = np.array(
        [preprocess_image(path) for path in batch_paths],
        dtype=np.float32
    )

    save_path = os.path.join(
        SAVE_DIR,
        f"test_batch_{batch_idx + 1}.npy"
    )

    np.save(save_path, batch_images)

    print(f"Saved {save_path} ({batch_images.shape[0]} images)")

    # Free RAM before processing next batch
    del batch_images

print("\nAll testing batches have been saved successfully!")

Total batches to create: 71
Processing Batch 1/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_1.npy (64 images)
Processing Batch 2/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_2.npy (64 images)
Processing Batch 3/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_3.npy (64 images)
Processing Batch 4/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_4.npy (64 images)
Processing Batch 5/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_5.npy (64 images)
Processing Batch 6/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_6.npy (64 images)
Processing Batch 7/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_7.npy (64 images)
Processing Batch 8/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_8.npy (64 images)
Processing Batch 9/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_9.npy (64 images)
Processing Batch 10/71...
Saved E:/FYP PROJECT/processed_test_batches\test_batch_10.npy (6

## Convert Labels into Numpy Arrays

In [122]:
y_train = np.array(y_train)

y_val = np.array(y_val)

y_test = np.array(y_test)

In [123]:
print("TEXT")
print(X_text_train.shape)
print(X_text_val.shape)
print(X_text_test.shape)

print("\nIMAGES")
print(X_img_train.shape)
print(X_img_val.shape)
print(X_img_test.shape)

print("\nMETADATA")
print(X_meta_train.shape)
print(X_meta_val.shape)
print(X_meta_test.shape)

print("\nLABELS")
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

TEXT
(21165, 599)
(4536, 599)
(4536, 599)

IMAGES
(21165,)
(4536,)
(4536,)

METADATA
(21165, 7)
(4536, 7)
(4536, 7)

LABELS
(21165,)
(4536,)
(4536,)


In [124]:
import os
import numpy as np

# Define the base path for saving in Google Drive
drive_path = r'E:/FYP PROJECT/'

# Define the specific folder for YouTube datasets
youtube_datasets_path = os.path.join(drive_path, 'youtube_datasets')

# Ensure the directory exists
os.makedirs(youtube_datasets_path, exist_ok=True)

print(f"Saving YouTube datasets to '{youtube_datasets_path}'...")

# Save text datasets
np.save(os.path.join(youtube_datasets_path, 'X_text_train.npy'), X_text_train)
np.save(os.path.join(youtube_datasets_path, 'X_text_val.npy'), X_text_val)
np.save(os.path.join(youtube_datasets_path, 'X_text_test.npy'), X_text_test)

# Save image datasets
np.save(os.path.join(youtube_datasets_path, 'X_img_train.npy'), X_img_train)
np.save(os.path.join(youtube_datasets_path, 'X_img_val.npy'), X_img_val)
np.save(os.path.join(youtube_datasets_path, 'X_img_test.npy'), X_img_test)

# Save metadata datasets
np.save(os.path.join(youtube_datasets_path, 'X_meta_train.npy'), X_meta_train)
np.save(os.path.join(youtube_datasets_path, 'X_meta_val.npy'), X_meta_val)
np.save(os.path.join(youtube_datasets_path, 'X_meta_test.npy'), X_meta_test)

# Save label datasets
np.save(os.path.join(youtube_datasets_path, 'y_train.npy'), y_train)
np.save(os.path.join(youtube_datasets_path, 'y_val.npy'), y_val)
np.save(os.path.join(youtube_datasets_path, 'y_test.npy'), y_test)

print("All YouTube datasets saved successfully.")

Saving YouTube datasets to 'E:/FYP PROJECT/youtube_datasets'...
All YouTube datasets saved successfully.
